In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load datasets
calendar = pd.read_csv("../data/calendar_clean.csv")
sell_prices = pd.read_csv("../data/sell_prices.csv")
sales = pd.read_csv("../data/sales_train_validation.csv")

print("Datasets loaded successfully!")
print("Calendar:", calendar.shape)
print("Sell Prices:", sell_prices.shape)
print("Sales:", sales.shape)

Datasets loaded successfully!
Calendar: (1969, 14)
Sell Prices: (6841121, 4)
Sales: (30490, 1919)


In [2]:
# Melt sales data
print("Converting to long format...")
sales_cols = [col for col in sales.columns if col.startswith('d_')]
info_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

sales_melted = sales[info_cols + sales_cols].melt(
    id_vars=info_cols,
    var_name='d',
    value_name='sales'
)

# Merge with calendar
sales_melted = sales_melted.merge(
    calendar[['d', 'date', 'month', 'year', 'weekday', 'wday',
              'event_name_1', 'snap_CA', 'snap_TX', 'snap_WI']],
    on='d', how='left'
)

# Take sample to avoid memory issues
sample = sales_melted.sample(n=500000, random_state=42).copy()

print("Done!")
print("Sample Shape:", sample.shape)
print("\nFirst 5 rows:")
print(sample.head())

Converting to long format...
Done!
Sample Shape: (500000, 17)

First 5 rows:
                                       id          item_id      dept_id  \
47561074      FOODS_3_548_WI_2_validation      FOODS_3_548      FOODS_3   
27190388      FOODS_3_231_WI_1_validation      FOODS_3_231      FOODS_3   
33395191      FOODS_3_319_CA_3_validation      FOODS_3_319      FOODS_3   
41582921  HOUSEHOLD_1_098_WI_2_validation  HOUSEHOLD_1_098  HOUSEHOLD_1   
50528035    HOBBIES_1_008_CA_3_validation    HOBBIES_1_008    HOBBIES_1   

             cat_id store_id state_id       d  sales        date  month  year  \
47561074      FOODS     WI_2       WI  d_1560      0  2015-05-07      5  2015   
27190388      FOODS     WI_1       WI   d_892      6  2013-07-08      7  2013   
33395191      FOODS     CA_3       CA  d_1096     31  2014-01-28      1  2014   
41582921  HOUSEHOLD     WI_2       WI  d_1364      0  2014-10-23     10  2014   
50528035    HOBBIES     CA_3       CA  d_1658      0  2015-08-13   

In [ ]:
# Add average price
avg_prices = sell_prices.groupby(['store_id', 'item_id'])['sell_price'].mean().reset_index()
avg_prices.columns = ['store_id', 'item_id', 'avg_price']
sample = sample.merge(avg_prices, on=['store_id', 'item_id'], how='left')

# Date features
sample['date'] = pd.to_datetime(sample['date'])
sample['day'] = sample['date'].dt.day
sample['week'] = sample['date'].dt.isocalendar().week.astype(int)
sample['quarter'] = sample['date'].dt.quarter
sample['is_weekend'] = sample['wday'].isin([1, 2]).astype(int)
sample['is_month_start'] = sample['date'].dt.is_month_start.astype(int)
sample['is_month_end'] = sample['date'].dt.is_month_end.astype(int)

# Event features
sample['has_event'] = (sample['event_name_1'] != 'No Event').astype(int)

# SNAP feature
sample['is_snap'] = 0
sample.loc[sample['state_id'] == 'CA', 'is_snap'] = sample.loc[sample['state_id'] == 'CA', 'snap_CA']
sample.loc[sample['state_id'] == 'TX', 'is_snap'] = sample.loc[sample['state_id'] == 'TX', 'snap_TX']
sample.loc[sample['state_id'] == 'WI', 'is_snap'] = sample.loc[sample['state_id'] == 'WI', 'snap_WI']

# Revenue feature
sample['revenue'] = sample['sales'] * sample['avg_price']

print("Features created!")
print("Shape:", sample.shape)
print("\nNew columns:")
print(['day', 'week', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'has_event', 'is_snap', 'revenue'])